# 01 · Prepare the ISEAR stories

This notebook turns the raw ISEAR file into the story table the experiment uses. It removes invalid answers, chooses the two countries with a rule fixed in advance, builds each writer's persona attributes, flags explicit nationality cues, exports a blind sample for hand-coding those flags, and computes the smallest difference the MVP can detect. Nothing here uses any model output, so everything in this notebook can be reported before the experiment is run.

In [1]:
import json, math, re, os
from pathlib import Path
import pandas as pd

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

## Settings

The data comes from the Hugging Face copy of the original ISEAR file (`savalera/isear-from-original`, the `original` split: all 7,666 stories). If the download fails, the notebook falls back to `data_isear_mirror.csv` in this folder. Cite the UNIGE source in the write-up.

In [2]:
HF_REPO = "savalera/isear-from-original"
LOCAL_MIRROR = "data_isear_mirror.csv"
OUT = Path("data"); OUT.mkdir(exist_ok=True)
N_COUNTRIES = 2                        # pre-registered: the two countries with most valid stories
CUE_SAMPLE_PER_COUNTRY = 60
SEED = 2026

## Codebook, exclusion rule and cue dictionary

The country, sex and field codes come from the ISEAR codebook; check them against the codebook distributed with the UNIGE file. The invalid-answer rule follows the exclusions listed by Turdubaeva & Lee (2026). The cue dictionary is itself an automated measurement, which is why a hand-coded sample is exported later.

In [3]:
COUNTRIES = {
    "1": "Sweden", "2": "Norway", "4": "Finland", "8": "Austria", "9": "Australia",
    "10": "Brazil", "12": "Bulgaria", "16": "New Zealand", "17": "the Netherlands",
    "19": "Spain", "20": "Zambia", "22": "the United States", "26": "India",
    "27": "Malawi", "30": "China", "33": "Honduras",
}
SEX = {"1": "man", "2": "woman"}                      # 0 = missing
FIELD = {"1": "psychology", "2": "the social sciences", "3": "languages",
         "4": "fine arts", "5": "law", "6": "natural sciences",
         "7": "engineering", "8": "medicine", "9": "another subject"}  # 0 = missing
EMOTIONS = ["joy", "fear", "anger", "sadness", "disgust", "shame", "guilt"]

INVALID = re.compile(
    r"^\[?\s*(?:nothing|none|no response|no answer|no description|not applicable|"
    r"not included|blank|n/?a\b|do not (know|remember|recall)|can'?t remember|"
    r"cannot (remember|recall)|can not (remember|recall|think)|never|i (have )?never|"
    r"i can ?not (recall|remember|think)|i do not (recall|remember)|same as|the same|as for|"
    r"doesn'?t apply)", re.I)

# Explicit nationality cues for the countries in the MVP. Extend before running.
# "english" is left out of "_any": in the ISEAR stories it almost always means the language
# or a class (44 of the Chinese hits), not a nationality cue.
CUES = {
    "Sweden": ["swed\\w*", "stockholm", "uppsala", "gothenburg", "g[oö]teborg", "malm[oö]",
               "lund", "ume[aå]", "krona", "kronor", "scandinavia\\w*", "nordic"],
    "Zambia": ["zambia\\w*", "lusaka", "kitwe", "ndola", "kabwe", "livingstone", "copperbelt",
               "kwacha", "ngwee", "bemba", "nyanja", "lozi", "tonga", "chewa", "nshima",
               "chitenge", "africa\\w*"],
    "China": ["china", "chinese", "beijing", "peking", "shanghai", "canton\\w*", "guangzhou",
              "yuan", "renminbi", "rmb", "mao", "communist party", "cultural revolution",
              "spring festival", "lunar new year", "taiwan\\w*", "hong kong", "korea\\w*",
              "japan\\w*", "opium war", "asia\\w*"],
    "_any": ["europe\\w*", "america\\w*", "british"],
}


## Power function

This is the minimum detectable difference for a paired binary outcome under a one-sided test, inflated by a design effect because each writer contributed several stories.

In [4]:
def mde(n_stories, n_writers, discordance, icc, alpha_z=1.645, power_z=0.842):
    """Minimum detectable difference for a paired binary outcome, one-sided test.

    discordance = share of stories whose correctness differs between the two conditions.
    icc         = correlation of outcomes between stories of the same writer.
    """
    m = n_stories / n_writers
    design_effect = 1 + (m - 1) * icc
    n_eff = n_stories / design_effect
    return (alpha_z + power_z) * math.sqrt(discordance / n_eff)


## 1. Load and clean

In [5]:
def load_isear():
    """All 7,666 ISEAR rows as strings. Hugging Face first, local mirror as a fallback."""
    try:
        from huggingface_hub import hf_hub_download
        parts = [pd.read_parquet(hf_hub_download(HF_REPO, f"original/{split}-00000-of-00001.parquet",
                                                 repo_type="dataset"))
                 for split in ("train", "validation", "test")]
        return pd.concat(parts, ignore_index=True).astype(str), f"huggingface:{HF_REPO}/original"
    except Exception as e:
        print("Hugging Face download failed, using the local mirror:", e)
        # 163 lines contain a '|' inside the story and are skipped by this reader, so the
        # mirror gives 751 Zambian and 551 Chinese stories instead of 770 and 553.
        return pd.read_csv(LOCAL_MIRROR, sep="|", dtype=str, on_bad_lines="skip"), LOCAL_MIRROR


raw, ISEAR_PATH = load_isear()
print(f"{len(raw)} rows from {ISEAR_PATH}")

df = raw[raw.COUN.isin(COUNTRIES)].copy()
df["country"] = df.COUN.map(COUNTRIES)
df["writer_id"] = df.COUN + "_" + df.SUBJ
df["story_id"] = df.MYKEY.fillna(df.ID)
df["emotion"] = df.Field1.str.strip().str.lower()
df["text"] = (df.SIT.fillna("").str.replace("á", "", regex=False)
              .str.replace(r"\s+", " ", regex=True).str.strip())
df["invalid"] = df.text.str.len().lt(5) | df.text.map(lambda t: bool(INVALID.search(t)))
df = df[df.emotion.isin(EMOTIONS)]
print(f"Loaded {len(df)} stories; removed {df.invalid.sum()} invalid answers.")
df[df.invalid].text.head(10).tolist()

7666 rows from huggingface:savalera/isear-from-original/original
Loaded 7666 stories; removed 160 invalid answers.


['No response.',
 'No response.',
 'Do not remember any incident.',
 'Cannot remember such a situation.',
 'The same situation as with sadness. I was afraid about what my parents would further do to distroy my relationship.',
 'No response.',
 'No response.',
 'I can not think of any incident when I have felt very ashamed. There are small incidents that I am ashamed of, for instance, not sending money to the children in Ethiopia.',
 'No response.',
 'No response.']

## 2. Choose the two countries

The rule is fixed before any model is run: take the countries with the most valid stories.

In [6]:
valid = df[~df.invalid].copy()
counts = valid.groupby("country").size().sort_values(ascending=False)
chosen = list(counts.index[:N_COUNTRIES])
print("Chosen countries:", chosen)
counts.to_frame("valid stories")

Chosen countries: ['Zambia', 'China']


,valid stories
country,
Zambia,770
China,553
Spain,541
Bulgaria,511
Sweden,502
Malawi,501
Finland,486
Austria,480
the Netherlands,479


## 3. Persona attributes and nationality cues

In [7]:
s = valid[valid.country.isin(chosen)].copy()
s["sex"] = s.SEX.map(SEX)
s["age"] = pd.to_numeric(s.AGE, errors="coerce")
s["field"] = s.FIEL.map(FIELD)
s["full_persona_complete"] = s[["sex", "age", "field"]].notna().all(axis=1)

pattern = {c: re.compile(r"\b(?:" + "|".join(CUES[c] + CUES["_any"]) + r")\b", re.I)
           for c in chosen if c in CUES}
missing = [c for c in chosen if c not in CUES]
if missing:
    print("WARNING: add cue words for", missing)
s["cue_flag"] = [bool(pattern[c].search(t)) if c in pattern else False
                 for c, t in zip(s.country, s.text)]

display(pd.crosstab(s.country, s.emotion))
display(s.groupby("country").agg(stories=("story_id", "size"), writers=("writer_id", "nunique"),
                                 cue_share=("cue_flag", "mean"), mean_age=("age", "mean")))

emotion,anger,disgust,fear,guilt,joy,sadness,shame
country,,,,,,,
China,79,79,79,79,79,79,79
Zambia,110,110,110,110,110,110,110


,stories,writers,cue_share,mean_age
country,,,,
China,553,79,0.072333,20.936709
Zambia,770,110,0.024675,22.354545


Look at which words trigger the cue flag. If the dictionary fires much more often in one country, its errors may differ by country too, and the hand-coded sample in step 5 tells you whether they do.

In [8]:
for c in chosen:
    if c in pattern:
        hits = s[s.country == c].text.map(lambda t: [x.lower() for x in pattern[c].findall(t)])
        print(c, pd.Series([w for l in hits for w in l]).value_counts().head(10).to_dict())

Zambia {'zambia': 8, 'lusaka': 6, 'kabwe': 2, 'copperbelt': 1, 'ndola': 1, 'african': 1, 'africa': 1, 'kitwe': 1}
China {'china': 30, 'asian': 11, 'chinese': 5, 'korea': 3, 'korean': 1, 'beijing': 1, 'opium war': 1, 'koreans': 1, 'americans': 1}


## 4. Save the story table

In [9]:
cols = ["story_id", "writer_id", "country", "emotion", "text", "sex", "age",
        "field", "full_persona_complete", "cue_flag"]
s[cols].to_csv(OUT / "stories.csv", index=False)
print("Saved", len(s), "stories to", OUT / "stories.csv")

Saved 1323 stories to data\stories.csv


## 5. Blind sample for hand-coding the cue flag

For each story, write 1 in `hand_cue` if a reader could guess the writer's country from the text, otherwise 0. The file deliberately hides the dictionary's own flag.

In [10]:
samp = s.groupby("country").sample(n=CUE_SAMPLE_PER_COUNTRY, random_state=SEED)
samp[["story_id", "country", "text"]].assign(hand_cue="").to_csv(OUT / "validation_cue_sample.csv", index=False)
print("Exported", len(samp), "stories to", OUT / "validation_cue_sample.csv")

Exported 120 stories to data\validation_cue_sample.csv


## 6. Smallest detectable difference

Report this range in the pre-registration, before the experiment is run.

In [11]:
n, w = len(s), s.writer_id.nunique()
power = {f"discordance={d}, icc={icc}": round(100 * mde(n, w, d, icc), 2)
         for d in (0.10, 0.20) for icc in (0.05, 0.10)}
summary = {"source_file": ISEAR_PATH, "countries": chosen,
           "stories_per_country": s.groupby("country").size().to_dict(),
           "writers_per_country": s.groupby("country").writer_id.nunique().to_dict(),
           "stories_total": n, "writers_total": w,
           "full_persona_complete": int(s.full_persona_complete.sum()),
           "cue_flagged": s.groupby("country").cue_flag.sum().to_dict(),
           "mde_percentage_points": power}
(OUT / "data_summary.json").write_text(json.dumps(summary, indent=2, default=int))
summary

{'source_file': 'huggingface:savalera/isear-from-original/original',
 'countries': ['Zambia', 'China'],
 'stories_per_country': {'China': 553, 'Zambia': 770},
 'writers_per_country': {'China': 79, 'Zambia': 110},
 'stories_total': 1323,
 'writers_total': 189,
 'full_persona_complete': 1302,
 'cue_flagged': {'China': 40, 'Zambia': 19},
 'mde_percentage_points': {'discordance=0.1, icc=0.05': 2.47,
  'discordance=0.1, icc=0.1': 2.73,
  'discordance=0.2, icc=0.05': 3.49,
  'discordance=0.2, icc=0.1': 3.87}}